#### Steps taken:
1. Read silver `results` table
2. Read silver `sprints` table
3. Add new column `session_type` with values `RACE` or `SPRINT`
4. UNION `results` and `sprints`
5. Derive additional columns
 - is_win -> Indicates that the driver own the race
 - is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
 - has_points -> Indicates that the driver has scored points
6. Write the transformed data to gold `fact_session_results` table

In [0]:
%run ../environment_config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog}.{gold_schema}.fact_session_results"

In [0]:
results_df = (spark.read
                   .table(f"{catalog}.{silver_schema}.results")
                   .withColumn("session_type", F.lit("Race"))
                   .drop("race_name", "race_date", "ingestion_timestamp", "souce_file", "source_path")
             )

In [0]:
sprints_df = (spark.read
                   .table(f"{catalog}.{silver_schema}.sprints")
                   .withColumn("session_type", F.lit("Sprint"))
                   .drop("race_name", "race_date", "ingestion_timestamp", "souce_file", "source_path")
)

In [0]:
union_result_sprint_df = results_df.unionByName(sprints_df)

In [0]:
final_result_sprints_df = union_result_sprint_df.withColumns({
                                   "is_win": F.col("final_position")==1,
                                   "is_podium": F.col("final_position").between(1,3),
                                   "has_points": F.col("points")>0                                             
                               })

In [0]:
display(final_result_sprints_df)

In [0]:
(
    final_result_sprints_df.write
                           .format("delta")
                           .mode("overwrite")
                           .saveAsTable(target_table)
)

In [0]:
# display(union_result_sprint_df.filter(F.col("season")==2025))

In [0]:
# display(spark.table("formula1.silver.results").filter(F.col("season")==2025))
# display(spark.table("formula1.silver.sprints").filter(F.col("season")==2025))